In [0]:
from pyspark.sql import SparkSession
print("Banking data lakehouse project started !")
print("Spark Version:", spark.version)
print("Python environment is ready")

Banking data lakehouse project started !
Spark Version: 4.2.0
Python environment is ready


In [0]:

from pyspark.sql.types import StructType, StructField, StringType, DoubleType

data = [
    ("TXN1001", "CUST01", "2026-10-01", 250.00, "CARD"),
    ("TXN1002", "CUST02", "2026-10-01", 1200.50, "UPI"),
    ("TXN1003", "CUST03", "2026-10-02", 75.25, "CARD"),
    ("TXN1004", "CUST01", "2026-10-02", 499.99, "BANK_TRANSFER"),
    ("TXN1005", "CUST04", "2026-10-03", 0.00, "UPI"),
    ("TXN1006", "CUST05", "not-a-date", 800.00, "CARD"),
    ("TXN1007", "", "2026-10-03", 120.00, "UPI"),
    ("TXN1008", "CUST06", "2026-10-04", -50.00, "CARD"),
    ("TXN1002", "CUST02", "2026-10-01", 1200.50, "UPI"),
    ("TXN1009", "CUST07", "2026-10-04", 350.75, "BANK_TRANSFER"),
    ("TXN1010", "CUST08", "2026-10-05", 90.00, "CARD"),
    ("TXN1011", "CUST09", "2026-10-05", 640.00, "UPI")
]

schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("transaction_date", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("transaction_type", StringType(), True)
])

raw_df = spark.createDataFrame(data, schema)

print("Total raw transactions:", raw_df.count())

display(raw_df)


Total raw transactions: 12


transaction_id,customer_id,transaction_date,amount,transaction_type
TXN1001,CUST01,2026-10-01,250.0,CARD
TXN1002,CUST02,2026-10-01,1200.5,UPI
TXN1003,CUST03,2026-10-02,75.25,CARD
TXN1004,CUST01,2026-10-02,499.99,BANK_TRANSFER
TXN1005,CUST04,2026-10-03,0.0,UPI
TXN1006,CUST05,not-a-date,800.0,CARD
TXN1007,,2026-10-03,120.0,UPI
TXN1008,CUST06,2026-10-04,-50.0,CARD
TXN1002,CUST02,2026-10-01,1200.5,UPI
TXN1009,CUST07,2026-10-04,350.75,BANK_TRANSFER


In [0]:
#step 5 create the bronze layer
bronze_table = "banking_bronze"

#preserve the original raw transactions 
(raw_df.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(bronze_table)
 )

print("Bronze Layer created successfully")
print("Bronze row count:", spark.table(bronze_table).count())

display(spark.table(bronze_table))

Bronze Layer created successfully
Bronze row count: 12


transaction_id,customer_id,transaction_date,amount,transaction_type
TXN1001,CUST01,2026-10-01,250.0,CARD
TXN1002,CUST02,2026-10-01,1200.5,UPI
TXN1003,CUST03,2026-10-02,75.25,CARD
TXN1004,CUST01,2026-10-02,499.99,BANK_TRANSFER
TXN1005,CUST04,2026-10-03,0.0,UPI
TXN1006,CUST05,not-a-date,800.0,CARD
TXN1007,,2026-10-03,120.0,UPI
TXN1008,CUST06,2026-10-04,-50.0,CARD
TXN1002,CUST02,2026-10-01,1200.5,UPI
TXN1009,CUST07,2026-10-04,350.75,BANK_TRANSFER


In [0]:

from pyspark.sql import functions as F

# Read raw data from the Bronze table
bronze_df = spark.table("banking_bronze")

# Clean and validate the records
prepared_df = (
    bronze_df
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("transaction_type", F.upper(F.trim(F.col("transaction_type"))))
    .withColumn(
        "parsed_date",
        F.expr("try_cast(transaction_date AS DATE)")
    )
    .withColumn(
        "rejection_reason",
        F.when(
            F.col("transaction_id").isNull() |
            (F.trim(F.col("transaction_id")) == ""),
            "Missing transaction ID"
        )
        .when(
            F.col("customer_id").isNull() |
            (F.col("customer_id") == ""),
            "Missing customer ID"
        )
        .when(
            F.col("parsed_date").isNull(),
            "Invalid transaction date"
        )
        .when(
            F.col("amount").isNull(),
            "Missing or invalid amount"
        )
        .when(
            F.col("amount") <= 0,
            "Amount must be positive"
        )
    )
)

# Separate invalid records
rejected_df = prepared_df.filter(
    F.col("rejection_reason").isNotNull()
)

# Keep only valid records
valid_df = prepared_df.filter(
    F.col("rejection_reason").isNull()
)

# Remove duplicate transaction IDs
deduplicated_df = valid_df.dropDuplicates(["transaction_id"])

# Select the final Silver columns
silver_df = deduplicated_df.select(
    "transaction_id",
    "customer_id",
    F.col("parsed_date").alias("transaction_date"),
    F.round("amount", 2).alias("amount"),
    "transaction_type"
)

# Save both Delta tables
(silver_df.write.format("delta")
    .mode("overwrite")
    .saveAsTable("banking_silver"))

(rejected_df.select(
    "transaction_id",
    "customer_id",
    "transaction_date",
    "amount",
    "transaction_type",
    "rejection_reason"
).write.format("delta")
    .mode("overwrite")
    .saveAsTable("banking_rejected"))

print("Silver layer created successfully!")
print("Clean transactions:", spark.table("banking_silver").count())
print("Rejected transactions:", spark.table("banking_rejected").count())

display(spark.table("banking_silver"))


Silver layer created successfully!
Clean transactions: 7
Rejected transactions: 4


transaction_id,customer_id,transaction_date,amount,transaction_type
TXN1003,CUST03,2026-10-02,75.25,CARD
TXN1004,CUST01,2026-10-02,499.99,BANK_TRANSFER
TXN1010,CUST08,2026-10-05,90.0,CARD
TXN1001,CUST01,2026-10-01,250.0,CARD
TXN1002,CUST02,2026-10-01,1200.5,UPI
TXN1011,CUST09,2026-10-05,640.0,UPI
TXN1009,CUST07,2026-10-04,350.75,BANK_TRANSFER


In [0]:

from pyspark.sql import functions as F

# Read clean transactions from Silver
silver_df = spark.table("banking_silver")

# Gold table 1: Payment type summary
payment_summary_df = (
    silver_df
    .groupBy("transaction_type")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount")
    )
    .orderBy("transaction_type")
)

(payment_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("banking_gold_payment_summary"))

# Gold table 2: Daily transaction summary
daily_summary_df = (
    silver_df
    .groupBy("transaction_date")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount")
    )
    .orderBy("transaction_date")
)

(daily_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("banking_gold_daily_summary"))

print("Gold layer created successfully!")

print("Payment type summary:")
display(spark.table("banking_gold_payment_summary"))

print("Daily transaction summary:")
display(spark.table("banking_gold_daily_summary"))


Gold layer created successfully!
Payment type summary:


transaction_type,transaction_count,total_amount
BANK_TRANSFER,2,850.74
CARD,3,415.25
UPI,2,1840.5


Daily transaction summary:


transaction_date,transaction_count,total_amount
2026-10-01,2,1450.5
2026-10-02,2,575.24
2026-10-04,1,350.75
2026-10-05,2,730.0


In [0]:

from pyspark.sql import functions as F

silver = spark.table("banking_silver")
rejected = spark.table("banking_rejected")
bronze = spark.table("banking_bronze")

print("=== DATA QUALITY VALIDATION ===")
print("Bronze rows:", bronze.count())
print("Silver rows:", silver.count())
print("Rejected rows:", rejected.count())

print("\n=== FINANCIAL TOTAL ===")
silver_total = silver.agg(
    F.round(F.sum("amount"), 2).alias("total")
).first()["total"]

gold_total = spark.table(
    "banking_gold_payment_summary"
).agg(
    F.round(F.sum("total_amount"), 2).alias("total")
).first()["total"]

print("Silver total:", silver_total)
print("Gold total:", gold_total)
print("Totals match:", silver_total == gold_total)

print("\n=== REJECTION REASONS ===")
display(
    rejected.groupBy("rejection_reason")
    .count()
    .orderBy(F.desc("count"))
)


=== DATA QUALITY VALIDATION ===
Bronze rows: 12
Silver rows: 7
Rejected rows: 4

=== FINANCIAL TOTAL ===
Silver total: 3106.49
Gold total: 3106.49
Totals match: True

=== REJECTION REASONS ===


rejection_reason,count
Amount must be positive,2
Invalid transaction date,1
Missing customer ID,1


In [0]:

# Query 1: Payment type summary
display(spark.sql("""
    SELECT
        transaction_type,
        transaction_count,
        total_amount
    FROM banking_gold_payment_summary
    ORDER BY total_amount DESC
"""))

# Query 2: Daily transaction trends
display(spark.sql("""
    SELECT
        transaction_date,
        transaction_count,
        total_amount
    FROM banking_gold_daily_summary
    ORDER BY transaction_date
"""))

# Query 3: Highest-value transactions
display(spark.sql("""
    SELECT
        transaction_id,
        customer_id,
        transaction_date,
        amount,
        transaction_type
    FROM banking_silver
    ORDER BY amount DESC
    LIMIT 5
"""))


transaction_type,transaction_count,total_amount
UPI,2,1840.5
BANK_TRANSFER,2,850.74
CARD,3,415.25


transaction_date,transaction_count,total_amount
2026-10-01,2,1450.5
2026-10-02,2,575.24
2026-10-04,1,350.75
2026-10-05,2,730.0


transaction_id,customer_id,transaction_date,amount,transaction_type
TXN1002,CUST02,2026-10-01,1200.5,UPI
TXN1011,CUST09,2026-10-05,640.0,UPI
TXN1004,CUST01,2026-10-02,499.99,BANK_TRANSFER
TXN1009,CUST07,2026-10-04,350.75,BANK_TRANSFER
TXN1001,CUST01,2026-10-01,250.0,CARD
